# Prétraitement — toutes les visites (V0, V1, V3, V5)

Exécutez ce notebook de haut en bas.  
Chaque section `### Vx` traite le fichier `Output/Vx.xlsx` et écrit le résultat dans `Output2/Vx.xlsx`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

---
## Fonction générique de traitement d'une visite

Cette fonction reçoit le **numéro de visite** (entier : 0, 1, 3 ou 5)  
et retourne un dictionnaire `{nom_feuille: DataFrame}` prêt à être écrit dans Excel.

In [ ]:
def traiter_visite(v, vc=False):
    chemin = f"Output/version_1/v{v}.xlsx"

    # ------------------------------------------------------------------
    # Feuille 1  →  DATES_VISITE  (uniquement V0 et VC)
    # ------------------------------------------------------------------
    if vc or v == 0:
        df_dates = pd.read_excel(chemin, sheet_name="DATES_VISITE")
        print(f"\nFeuille DATES_VISITE : {df_dates.shape}")

        # Renommage selon la visite
        if vc:
            df_dates.rename(columns={"D_CHIR": "DATE"}, inplace=True)

    # ------------------------------------------------------------------
    # Feuille 3  →  df_LEDD  (commun à tous)
    # ------------------------------------------------------------------
    df_feuil1 = pd.read_excel(chemin, sheet_name="Feuil1")
    print(f"\nFeuille Feuil1 (LEDD) : {df_feuil1.shape}")

    df_feuil1.rename(
        columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True
    )
    df_LEDD = df_feuil1.copy()
    df_LEDD.drop("Num", axis=1, inplace=True)
    df_LEDD.insert(0, "VISITE", v)
    print(f"df_LEDD              : {df_LEDD.shape}")

    # ------------------------------------------------------------------
    # Feuille 4  →  df_LEDD_info  (commun à tous)
    # ------------------------------------------------------------------
    df_ledd = pd.read_excel(chemin, sheet_name="LEDD")
    print(f"\nFeuille LEDD         : {df_ledd.shape}")

    df_LEDD_info = df_ledd.iloc[:, :-7].copy()
    df_LEDD_info.drop("v", axis=1, inplace=True)
    df_LEDD_info.rename(
        columns={"Subject Identifier for the Study": "SUBJID"}, inplace=True
    )
    df_LEDD_info.insert(0, "VISITE", v)
    df_LEDD_info.drop("Num", axis=1, inplace=True)
    df_LEDD_info.drop("Visit", axis=1, inplace=True)
    print(f"df_LEDD_info         : {df_LEDD_info.shape}")

    # ------------------------------------------------------------------
    # Feuille 5  →  df_PSYCHOTROPES  (commun à tous)
    # ------------------------------------------------------------------
    df_PSYCHOTROPES = pd.read_excel(chemin, sheet_name="PSYCHOTROPES")
    print(f"\nFeuille PSYCHOTROPES : {df_PSYCHOTROPES.shape}")

    df_PSYCHOTROPES.insert(0, "VISITE", v)
    df_PSYCHOTROPES.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    df_PSYCHOTROPES.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    print(f"df_PSYCHOTROPES      : {df_PSYCHOTROPES.shape}")

    # ------------------------------------------------------------------
    # Feuille 6  →  df_AUTRE_PARKINSON  (commun à tous)
    # ------------------------------------------------------------------
    df_AUTRE_PARKINSON = pd.read_excel(chemin, sheet_name="AUTRE_PARKINSON")
    print(f"\nFeuille AUTRE_PARKINSON : {df_AUTRE_PARKINSON.shape}")

    df_AUTRE_PARKINSON.insert(0, "VISITE", v)
    df_AUTRE_PARKINSON.drop(columns=["NUM", "Unnamed: 3"], inplace=True)
    df_AUTRE_PARKINSON.drop(columns=["VISIT", "INIT_PAT"], inplace=True)
    print(f"df_AUTRE_PARKINSON   : {df_AUTRE_PARKINSON.shape}")

    # ------------------------------------------------------------------
    # Feuille 7  →  df_CONSO_SPECIFIQUE  (commun à tous)
    # ------------------------------------------------------------------
    df_CONSO_SPECIFIQUE = pd.read_excel(chemin, sheet_name="CONSO_SPECIFIQUE")
    print(f"\nFeuille CONSO_SPECIFIQUE : {df_CONSO_SPECIFIQUE.shape}")

    df_CONSO_SPECIFIQUE.insert(0, "VISITE", v)
    df_CONSO_SPECIFIQUE.drop(
        columns=["NUM", "VISIT_NOM", "NUM_CENTRE", "NUM_PAT", "INIT_PAT"],
        inplace=True,
    )
    df_CONSO_SPECIFIQUE.drop("VISIT", axis=1, inplace=True)
    print(f"df_CONSO_SPECIFIQUE  : {df_CONSO_SPECIFIQUE.shape}")

    # ------------------------------------------------------------------
    # Retour VC  →  seulement les feuilles communes
    # ------------------------------------------------------------------
    if vc:
        return {
            "Vc"               : df_dates,
            "LEDD_info"        : df_LEDD_info,
            "LEDD"             : df_LEDD,
            "CONSO_SPECIFIQUE" : df_CONSO_SPECIFIQUE,
            "PSYCHOTROPES"     : df_PSYCHOTROPES,
            "AUTRE_PARKINSON"  : df_AUTRE_PARKINSON,
        }

    # ------------------------------------------------------------------
    # Suite  →  uniquement pour V0 à V5
    # ------------------------------------------------------------------
    df_base = pd.read_excel(chemin, sheet_name=f"df_v{v}")
    df_base.drop(columns=["POIDS_NR", "TAILLE_NR", "TITRE"], inplace=True)

    # Fusion avec df_dates uniquement pour V0
    if v == 0:
        df_V = pd.merge(df_dates, df_base, on="SUBJID", how="left")
        df_V.rename(columns={"D_SCREEN": "DATE"}, inplace=True)
    else:
        df_V = df_base.copy()

    df_V.insert(0, "VISITE", v)
    df_V.drop("INIT_PAT", axis=1, inplace=True)
    print(f"df_V (après fusion)  : {df_V.shape}")

    df_DIGITSMT = pd.read_excel(chemin, sheet_name="DIGITSMT_TRAILMT_DKEFS")
    print(f"\nFeuille DIGITSMT_TRAILMT_DKEFS : {df_DIGITSMT.shape}")
    df_DIGITSMT.insert(0, "VISITE", v)
    df_DIGITSMT.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_PDQ39 = pd.read_excel(chemin, sheet_name="PDQ39")
    df_PDQ39.insert(0, "VISITE", v)
    df_PDQ39.drop(columns=["VISIT", "INIT_PAT"], inplace=True)

    df_LARS = pd.read_excel(chemin, sheet_name="LARS")
    df_LARS.insert(0, "VISITE", v)
    df_LARS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_HAMA = pd.read_excel(chemin, sheet_name="HAMA")
    df_HAMA.insert(0, "VISITE", v)
    df_HAMA.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_HAMD = pd.read_excel(chemin, sheet_name="HAMD")
    df_HAMD.insert(0, "VISITE", v)
    df_HAMD.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_MOCA = pd.read_excel(chemin, sheet_name="MOCA")
    df_MOCA.insert(0, "VISITE", v)
    df_MOCA.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_QUIP = pd.read_excel(chemin, sheet_name="QUIP")
    df_QUIP.insert(0, "VISITE", v)
    df_QUIP.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_ECMP = pd.read_excel(chemin, sheet_name="ECMP")
    df_ECMP.insert(0, "VISITE", v)
    df_ECMP.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    df_UPDRS = pd.read_excel(chemin, sheet_name="UPDRS")
    df_UPDRS.insert(0, "VISITE", v)
    df_UPDRS.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    if v in (3, 5):
        df_UPDRSIII_COMPLET = pd.read_excel(chemin, sheet_name="UPDRSIII_COMPLET")
        df_UPDRSIII_COMPLET.insert(0, "VISITE", v)
        df_UPDRSIII_COMPLET.drop(columns=["TITRE", "INIT_PAT"], inplace=True)

    if v in (1, 2, 3):
        df_FREQUENCE = pd.read_excel(chemin, sheet_name="FREQUENCE")
        df_FREQUENCE.drop(columns=["VISITE", "INIT_PAT"], inplace=True)
        df_FREQUENCE.insert(0, "VISITE", v)

    # ------------------------------------------------------------------
    # Retour V0 à V5
    # ------------------------------------------------------------------
    result = {
        f"V{v}"                  : df_V,
        "LEDD_info"              : df_LEDD_info,
        "LEDD"                   : df_LEDD,
        "CONSO_SPECIFIQUE"       : df_CONSO_SPECIFIQUE,
        "PSYCHOTROPES"           : df_PSYCHOTROPES,
        "AUTRE_PARKINSON"        : df_AUTRE_PARKINSON,
        "UPDRSIV"                : df_UPDRS,
        "PDQ39"                  : df_PDQ39,
        "QUIP"                   : df_QUIP,
        "MOCA"                   : df_MOCA,
        "HAMA"                   : df_HAMA,
        "HAMD"                   : df_HAMD,
        "LARS"                   : df_LARS,
        "ECMP"                   : df_ECMP,
        "DIGITSMT_TRAILMT_DKEFS" : df_DIGITSMT,
    }

    if v in (3, 5):
        result["UPDRSIII"] = df_UPDRSIII_COMPLET

    if v in (1, 2, 3):
        result["FREQUENCE"] = df_FREQUENCE

    return result

In [ ]:
ORDRE_FEUILLES = [
    "LEDD_info",
    "LEDD",
    "CONSO_SPECIFIQUE",
    "PSYCHOTROPES",
    "AUTRE_PARKINSON",
    "UPDRSIII",          
    "UPDRSIII_TOTAUX",   
    "UPDRSIV",
    "PDQ39",
    "QUIP",
    "MOCA",
    "HAMA",
    "HAMD",
    "LARS",
    "ECMP",
    "DIGITSMT_TRAILMT_DKEFS",
    "FREQUENCE",
]

def reordonner_feuilles(sheets, v):
    cle_visite    = f"V{v}"
    ordre_complet = [cle_visite] + ORDRE_FEUILLES
    return {k: sheets[k] for k in ordre_complet if k in sheets}

In [ ]:
def charger_updrs_iii():
    """
    Lit le fichier UPDRS III et sépare les colonnes V0 / V1.
    Retourne (df_v0_UPDRS_III, df_v1_UPDRS_III,
              df_v0_UPDRSIII_TOTAUX, df_v1_UPDRSIII_TOTAUX,
              df_v3_UPDRSIII_TOTAUX, df_v5_UPDRSIII_TOTAUX)
    """
    chemin = "Data/Matthieu_Soumaya_Dec2025.xlsx"

    # ---- UPDRSIII complet ----
    df_III = pd.read_excel(chemin, sheet_name="UPDRSIII_COMPLET_V0_V1")
    # print(f"UPDRSIII_COMPLET_V0_V1 : {df_III.shape}")

    # Colonnes 1-263 → V0 ;  colonnes 264+ → V1
    df_v0_III = pd.concat([df_III.iloc[:, [0]], df_III.iloc[:, 1:264]], axis=1)
    df_v1_III = pd.concat([df_III.iloc[:, [0]], df_III.iloc[:, 264:]], axis=1)
    df_v0_III.insert(0, "VISITE", 0)
    df_v1_III.insert(0, "VISITE", 1)

    # ---- UPDRSIII totaux ----
    df_tot = pd.read_excel(chemin, sheet_name="UPDRSIII_TOTAUX ")
    print(f"UPDRSIII_TOTAUX        : {df_tot.shape}")

    # Repérage dynamique de la colonne séparatrice "Unnamed: 8"
    sep_col = df_tot.columns.get_loc("Unnamed: 8")  # = 8

    df_v0_tot  = pd.concat([df_tot.iloc[:, [0]], df_tot.iloc[:, 1:sep_col]], axis=1)
    df_v1_tot  = pd.concat([df_tot.iloc[:, [0]], df_tot.iloc[:, sep_col+1:sep_col+5]], axis=1)
    df_v3_tot  = df_tot[["SUBJID", "ON_TOTAL_V3"]].copy()
    df_v5_tot  = df_tot[["SUBJID", "ON_TOTAL_V5"]].copy()


    df_v0_tot.insert(0, "VISITE", 0) 
    df_v1_tot.insert(0, "VISITE", 1)  
    df_v3_tot.insert(0, "VISITE", 3)  
    df_v5_tot.insert(0, "VISITE", 5)  

    return df_v0_III, df_v1_III, df_v0_tot, df_v1_tot, df_v3_tot, df_v5_tot


df_v0_UPDRS_III, df_v1_UPDRS_III, \
df_v0_UPDRSIII_TOTAUX, df_v1_UPDRSIII_TOTAUX, \
df_v3_UPDRSIII_TOTAUX, df_v5_UPDRSIII_TOTAUX = charger_updrs_iii()

---
## Traitement + écriture de chaque visite

### V0

In [ ]:
sheets_V0 = traiter_visite(0)

# Ajout des feuilles UPDRS III propres à V0
sheets_V0["UPDRSIII"]        = df_v0_UPDRS_III
sheets_V0["UPDRSIII_TOTAUX"] = df_v0_UPDRSIII_TOTAUX
sheets_V0 = reordonner_feuilles(sheets_V0, 0)
# Écriture
OUTPUT_DIR = "Output/version_2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V0.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V0.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V0.xlsx  →  {len(sheets_V0)} feuilles")

### V1

In [ ]:
sheets_V1 = traiter_visite(1)

# Ajout des feuilles UPDRS III propres à V1
sheets_V1["UPDRSIII"]        = df_v1_UPDRS_III
sheets_V1["UPDRSIII_TOTAUX"] = df_v1_UPDRSIII_TOTAUX
sheets_V1 = reordonner_feuilles(sheets_V1, 1)
OUTPUT_DIR = "Output/version_2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V1.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V1.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V1.xlsx  →  {len(sheets_V1)} feuilles")

### V3

In [ ]:
sheets_V3 = traiter_visite(3)

# UPDRS III totaux pour V3 (colonne ON_TOTAL_V3 uniquement)
sheets_V3["UPDRSIII_TOTAUX"] = df_v3_UPDRSIII_TOTAUX
sheets_V3 = reordonner_feuilles(sheets_V3, 3)
OUTPUT_DIR = "Output/version_2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V3.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V3.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output/version_2/V3.xlsx  →  {len(sheets_V3)} feuilles")

### V5

In [ ]:
sheets_V5 = traiter_visite(5)

# UPDRS III totaux pour V5 (colonne ON_TOTAL_V5 uniquement)
sheets_V5["UPDRSIII_TOTAUX"] = df_v5_UPDRSIII_TOTAUX
sheets_V5 = reordonner_feuilles(sheets_V5, 5)
OUTPUT_DIR = "Output/version_2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/V5.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_V5.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/V5.xlsx  →  {len(sheets_V5)} feuilles")

### Vc 

In [ ]:
sheets_Vc = traiter_visite("c", vc=True)
sheets_Vc = reordonner_feuilles(sheets_Vc, "c")

OUTPUT_DIR = "Output/version_2"
with pd.ExcelWriter(f"{OUTPUT_DIR}/Vc.xlsx", engine="openpyxl") as writer:
    for sheet_name, df in sheets_Vc.items():
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
print(f"[OK] Output2/Vc.xlsx  →  {len(sheets_Vc)} feuilles")

# Prétraitement info statiques 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
xl = pd.ExcelFile("Output/version_1/info_statiques.xlsx")

print(f"Nombre de feuille : ",len(xl.sheet_names))
# une par ligne
for nom in xl.sheet_names:
    print(nom)

In [ ]:
df_feuille1 = pd.read_excel("Output/version_1/info_statiques.xlsx", sheet_name="Sheet1")
print(df_feuille1.shape)
df_feuille1.head()

In [ ]:
df_feuille2 = pd.read_excel("Output/version_1/info_statiques.xlsx", sheet_name="Sheet2")
print(df_feuille2.shape)
df_feuille2.head()

In [ ]:
df_Statiques = pd.merge(df_feuille1,df_feuille2,on="SUBJID",how="left")
df_Statiques.head() 

In [ ]:
df_Statiques.to_excel("Output/version_2/info.xlsx",index=False)